In [ ]:
import os
import json
import warnings
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_PATH = "./training_data_normalized.csv"
OUT_BASE = "./dag_out/NOTEARS"
RANDOM_STATE = 42
MAX_FEATURES = None

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

def load_numeric_X(
    data_path: str,
    drop_target_candidates: bool = True,
    max_features: Optional[int] = None,
    random_state: int = 42
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    np.random.seed(random_state)
    df = pd.read_csv(data_path, low_memory=False)

    if drop_target_candidates:
        drop_cols = [c for c in df.columns if c in TARGET_CANDIDATES]
        if drop_cols:
            print(f"[INFO] drop target candidates: {drop_cols}")
            df = df.drop(columns=drop_cols)

    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    if max_features is not None and df.shape[1] > max_features:
        df = df.iloc[:, :max_features].copy()
        print(f"[INFO] feature capped: {max_features}")

    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    X = df.values.astype(float)
    X = X - X.mean(axis=0, keepdims=True)
    print(f"[INFO] X shape: {X.shape}")
    return df, X, col_names

def break_cycles_by_removing_small_edges(W: np.ndarray) -> np.ndarray:
    W2 = W.copy()

    def build_graph(Wm):
        G = {i: [] for i in range(Wm.shape[0])}
        for i in range(Wm.shape[0]):
            for j in range(Wm.shape[1]):
                if i != j and abs(Wm[i, j]) > 0:
                    G[i].append(j)
        return G

    def find_cycle_edges(G):
        d = len(G)
        color = [0] * d
        parent = [-1] * d

        def dfs(u):
            color[u] = 1
            for v in G[u]:
                if color[v] == 0:
                    parent[v] = u
                    cyc = dfs(v)
                    if cyc is not None:
                        return cyc
                elif color[v] == 1:
                    nodes = [v]
                    cur = u
                    while cur != v and cur != -1:
                        nodes.append(cur)
                        cur = parent[cur]
                    nodes.append(v)
                    nodes = nodes[::-1]
                    return [(a, b) for a, b in zip(nodes[:-1], nodes[1:])]
            color[u] = 2
            return None

        for s in range(d):
            if color[s] == 0:
                cyc = dfs(s)
                if cyc is not None:
                    return cyc
        return None

    while True:
        cyc = find_cycle_edges(build_graph(W2))
        if cyc is None:
            break
        mags = [(abs(W2[i, j]), i, j) for (i, j) in cyc]
        mags.sort(key=lambda x: x[0])
        _, i_min, j_min = mags[0]
        W2[i_min, j_min] = 0.0

    return W2

def save_artifacts(W: np.ndarray, col_names: List[str], out_dir: str, alg_name: str) -> None:
    import networkx as nx

    os.makedirs(out_dir, exist_ok=True)

    edges = []
    for i, src in enumerate(col_names):
        for j, tgt in enumerate(col_names):
            if i != j and abs(W[i, j]) > 0:
                edges.append([src, tgt, float(W[i, j])])

    edge_df = pd.DataFrame(edges, columns=["source", "target", "weight"])
    edge_path = os.path.join(out_dir, f"edges_{alg_name}.csv")
    edge_df.to_csv(edge_path, index=False)

    adj_df = pd.DataFrame(W, index=col_names, columns=col_names)
    adj_path = os.path.join(out_dir, f"adj_{alg_name}.csv")
    adj_df.to_csv(adj_path)

    G = nx.DiGraph()
    for n in col_names:
        G.add_node(n)
    for _, r in edge_df.iterrows():
        G.add_edge(r["source"], r["target"], weight=float(r["weight"]))

    graphml_path = os.path.join(out_dir, f"graph_{alg_name}.graphml")
    gexf_path = os.path.join(out_dir, f"graph_{alg_name}.gexf")
    nx.write_graphml(G, graphml_path)
    nx.write_gexf(G, gexf_path)

    nodes = [{"id": n} for n in G.nodes()]
    jedges = [{"source": u, "target": v, "weight": float(G[u][v].get("weight", 0.0))} for u, v in G.edges()]
    json_path = os.path.join(out_dir, f"graph_{alg_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"nodes": nodes, "edges": jedges}, f, ensure_ascii=False, indent=2)

    print(f"[SAVE] {alg_name}")
    print(f"  - {edge_path} (n_edges={len(edge_df)})")
    print(f"  - {adj_path}")
    print(f"  - {graphml_path}")
    print(f"  - {gexf_path}")
    print(f"  - {json_path}")

def main():
    _, X, col_names = load_numeric_X(DATA_PATH, drop_target_candidates=True, max_features=MAX_FEATURES, random_state=RANDOM_STATE)

    try:
        from cdt.causality.graph import NOTears
    except Exception:
        print("[ERROR] NOTEARS requires 'cdt'. Install: pip install cdt")
        raise

    model = NOTears()
    W_df = model.predict(pd.DataFrame(X, columns=col_names))
    W = W_df.to_numpy(dtype=float)

    W = break_cycles_by_removing_small_edges(W)

    save_artifacts(W, col_names, OUT_BASE, "NOTEARS")

if __name__ == "__main__":
    main()


[INFO] drop target candidates: ['label']
[INFO] X shape: (17881, 14)
[ERROR] NOTEARS requires 'cdt'. Install: pip install cdt


ImportError: cannot import name 'NOTears' from 'cdt.causality.graph' (c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\cdt\causality\graph\__init__.py)